In [0]:
from pyspark.sql import functions as F

In [0]:
# silver_vaccinations
SOURCE_CATALOG_NAME_SV = 'covid_19'
SOURCE_SCHEMA_NAME_SV = 'silver'
SOURCE_TABLE_NAME_SV = 'vaccinations'

# dim_country
SOURCE_CATALOG_NAME_DC = 'covid_19'
SOURCE_SCHEMA_NAME_DC = 'gold'
SOURCE_TABLE_NAME_DC = 'dim_country'

# dim_date
SOURCE_CATALOG_NAME_DT = 'covid_19'
SOURCE_SCHEMA_NAME_DT = 'gold'
SOURCE_TABLE_NAME_DT = 'dim_date'

# fact_vaccinations
TARGET_CATALOG_NAME = 'covid_19'
TARGET_SCHEMA_NAME = 'gold'
TARGET_TABLE_NAME = 'fact_vaccinations'

In [0]:
df_silver_vaccinations = spark.table(f'{SOURCE_CATALOG_NAME_SV}.{SOURCE_SCHEMA_NAME_SV}.{SOURCE_TABLE_NAME_SV}')
df_dim_country = spark.table(f'{SOURCE_CATALOG_NAME_DC}.{SOURCE_SCHEMA_NAME_DC}.{SOURCE_TABLE_NAME_DC}')
df_dim_date = spark.table(f'{SOURCE_CATALOG_NAME_DT}.{SOURCE_SCHEMA_NAME_DT}.{SOURCE_TABLE_NAME_DT}')

In [0]:
df_fact_vaccinations = (
    df_silver_vaccinations.alias("sv")
    .join(
        df_dim_country.alias("dc"),
        on=["country", "iso_code"],
        how="inner"
    )
    .join(
        df_dim_date.alias("dt"),
        on="date",
        how="inner"
    )
    .select(
        "dc.country_key",
        "dt.date_key",

        "sv.total_vaccinations",
        "sv.people_vaccinated",
        "sv.people_fully_vaccinated",
        "sv.total_boosters",
        "sv.daily_vaccinations",
        "sv.daily_people_vaccinated"
    )
)

In [0]:
df_fact_vaccinations\
    .write\
    .mode('overwrite')\
    .saveAsTable(f'{TARGET_CATALOG_NAME}.{TARGET_SCHEMA_NAME}.{TARGET_TABLE_NAME}')